In [1]:
import ee
ee.Authenticate()
ee.Initialize(project='earth-engine-uhi')  # Pune ID-ul exact copiat la Pasul 2

print("Conexiunea la Earth Engine funcționează perfect!")

Conexiunea la Earth Engine funcționează perfect!


In [2]:
import ee

# 1. Definim punctul central (București) și un buffer de analiză
bucuresti_center = ee.Geometry.Point([26.1025, 44.4268])

# 2. Interogăm colecția Landsat 8/9 Level-2 (Termic - LST)
# Filtrăm: Iunie - August 2023, nori sub 20%
landsat_summer = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                  .filterBounds(bucuresti_center)
                  .filterDate('2023-06-01', '2023-08-31')
                  .filter(ee.Filter.lt('CLOUD_COVER', 20)))

# 3. Interogăm colecția Sentinel-2 Level-2A (Optic - NDVI/NDBI)
# Același interval temporal
sentinel_summer = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                   .filterBounds(bucuresti_center)
                   .filterDate('2023-06-01', '2023-08-31')
                   .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

# Extragem metadatele ca dovadă de acces
l8_dates = landsat_summer.aggregate_array('DATE_ACQUIRED').getInfo()
l8_count = landsat_summer.size().getInfo()
s2_count = sentinel_summer.size().getInfo()

print("=" * 60)
print(f"DOVADĂ ACCESARE SURSA 1 (Landsat 8 - LST):")
print(f"  • Număr scene senine identificate: {l8_count}")
print(f"  • Datele de achiziție: {l8_dates}")
print("=" * 60)
print(f"DOVADĂ ACCESARE SURSA 1 (Sentinel-2 - Optic):")
print(f"  • Număr scene senine identificate: {s2_count}")
print("=" * 60)

DOVADĂ ACCESARE SURSA 1 (Landsat 8 - LST):
  • Număr scene senine identificate: 7
  • Datele de achiziție: ['2023-06-06', '2023-06-22', '2023-07-24', '2023-08-09', '2023-08-25', '2023-06-29', '2023-07-15']
DOVADĂ ACCESARE SURSA 1 (Sentinel-2 - Optic):
  • Număr scene senine identificate: 13


In [3]:
import ee
import pandas as pd

# 1. Punct central București
bucuresti_center = ee.Geometry.Point([26.1025, 44.4268])

# 2. Colecții filtrate (Vara 2023)
landsat_summer = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                  .filterBounds(bucuresti_center)
                  .filterDate('2023-06-01', '2023-08-31')
                  .filter(ee.Filter.lt('CLOUD_COVER', 20)))

sentinel_summer = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                   .filterBounds(bucuresti_center)
                   .filterDate('2023-06-01', '2023-08-31')
                   .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

# 3. Funcții helper pentru extragerea proprietăților tabulare
def get_landsat_metadata(img):
    return ee.Feature(None, {
        'Satelit': 'Landsat 8 (TIRS/OLI)',
        'Tip_Date': 'Termic (LST) & Refl.',
        'Data_Achizitie': img.get('DATE_ACQUIRED'),
        'Acoperire_Nori_%': ee.Number(img.get('CLOUD_COVER')).round(),
        'Scene_ID': img.get('system:index')
    })

def get_sentinel_metadata(img):
    # Sentinel stochează data în timestamp milisecunde
    date_str = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
    return ee.Feature(None, {
        'Satelit': 'Sentinel-2 (MSI)',
        'Tip_Date': 'Optic (NDVI / NDBI)',
        'Data_Achizitie': date_str,
        'Acoperire_Nori_%': ee.Number(img.get('CLOUDY_PIXEL_PERCENTAGE')).round(),
        'Scene_ID': img.get('system:index')
    })

# 4. Mapare și extragere în Python
l8_features = landsat_summer.map(get_landsat_metadata).getInfo()['features']
s2_features = sentinel_summer.map(get_sentinel_metadata).getInfo()['features']

# 5. Construire DataFrame
rows = [f['properties'] for f in l8_features] + [f['properties'] for f in s2_features]
df_scene = pd.DataFrame(rows)

# Sortare cronologică
df_scene['Data_Achizitie'] = pd.to_datetime(df_scene['Data_Achizitie'])
df_scene = df_scene.sort_values(by=['Data_Achizitie', 'Satelit']).reset_index(drop=True)

# Reordonare coloane
df_scene = df_scene[['Data_Achizitie', 'Satelit', 'Tip_Date', 'Acoperire_Nori_%', 'Scene_ID']]

# Afișare tabel formatat în notebook
display(df_scene)

,Data_Achizitie,Satelit,Tip_Date,Acoperire_Nori_%,Scene_ID
0,2023-06-06,Landsat 8 (TIRS/OLI),Termic (LST) & Refl.,7,LC08_182029_20230606
1,2023-06-22,Landsat 8 (TIRS/OLI),Termic (LST) & Refl.,9,LC08_182029_20230622
2,2023-06-24,Sentinel-2 (MSI),Optic (NDVI / NDBI),1,20230624T090601_20230624T091055_T35TMK
3,2023-06-29,Landsat 8 (TIRS/OLI),Termic (LST) & Refl.,13,LC08_183029_20230629
4,2023-06-29,Sentinel-2 (MSI),Optic (NDVI / NDBI),9,20230629T090559_20230629T091459_T35TMK
5,2023-07-04,Sentinel-2 (MSI),Optic (NDVI / NDBI),12,20230704T090601_20230704T091054_T35TMK
6,2023-07-09,Sentinel-2 (MSI),Optic (NDVI / NDBI),4,20230709T090559_20230709T091057_T35TMK
7,2023-07-14,Sentinel-2 (MSI),Optic (NDVI / NDBI),0,20230714T090601_20230714T091057_T35TMK
8,2023-07-15,Landsat 8 (TIRS/OLI),Termic (LST) & Refl.,15,LC08_183029_20230715
9,2023-07-19,Sentinel-2 (MSI),Optic (NDVI / NDBI),17,20230719T090559_20230719T091058_T35TMK


In [4]:
import ee

ee.Initialize(project='earth-engine-uhi')

# 1. Punct de control: Piața Universității, București
point_sample = ee.Geometry.Point([26.1025, 44.4355])

# 2. Sursa 3: WorldPop 100m (Populație)
worldpop = (ee.ImageCollection('WorldPop/GP/100m/pop')
            .filterBounds(point_sample)
            .filter(ee.Filter.eq('year', 2020))
            .first())

pop_val = worldpop.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point_sample,
    scale=100
).getInfo()

# 3. Sursa 2b: GHSL Built-Up Grid (Copernicus JRC Built Surface Presence)
ghsl_built = ee.Image('JRC/GHSL/P2016/BUILT_LDSMT_GLOBE_V1')

built_val = ghsl_built.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point_sample,
    scale=100
).getInfo()

print("=" * 60)
print("DOVADĂ ACCESARE SURSA 3 (WorldPop 100m - Populație):")
print(f"  • Asset ID: WorldPop/GP/100m/pop (anul 2020)")
print(f"  • Estimare populație per celulă 100m: {pop_val.get('population'):.1f} locuitori")
print("=" * 60)
print("DOVADĂ ACCESARE SURSA 2b (GHSL Urban Built-Up):")
print(f"  • Asset ID: JRC/GHSL/P2016/BUILT_LDSMT_GLOBE_V1")
print(f"  • Valoare grid construit (epoch classification): {built_val}")
print("=" * 60)

/usr/local/lib/python3.13/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for JRC/GHSL/P2016/BUILT_LDSMT_GLOBE_V1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by JRC/GHSL/P2023A/GHS_BUILT_S

Learn more: https://developers.google.com/earth-engine/datasets/catalog/JRC_GHSL_P2016_BUILT_LDSMT_GLOBE_V1

  warnings.warn(warning, category=DeprecationWarning)


DOVADĂ ACCESARE SURSA 3 (WorldPop 100m - Populație):
  • Asset ID: WorldPop/GP/100m/pop (anul 2020)
  • Estimare populație per celulă 100m: 101.5 locuitori
DOVADĂ ACCESARE SURSA 2b (GHSL Urban Built-Up):
  • Asset ID: JRC/GHSL/P2016/BUILT_LDSMT_GLOBE_V1
  • Valoare grid construit (epoch classification): {'built': 6, 'cnfd': 236, 'dm': 16}


# Data Card

In [5]:
import ee

ee.Initialize(project='earth-engine-uhi')

def inspect_gee_asset(asset_id: str, is_collection: bool = False):
    if is_collection:
        img = ee.ImageCollection(asset_id).first()
    else:
        img = ee.Image(asset_id)
        
    info = img.getInfo()
    bands = [b['id'] for b in info.get('bands', [])]
    
    # Selectăm prima bandă și îi extragem proiecția
    first_band = img.select(0)
    proj = first_band.projection()
    
    # Metoda corectă în EE Python API este nominalScale()
    nominal_scale = proj.nominalScale().getInfo()
    crs = proj.crs().getInfo()
    
    print(f"Dataset: {asset_id}")
    print(f"  • Număr total benzi: {len(bands)}")
    print(f"  • Primele benzi: {bands[:6]}")
    print(f"  • Rezoluție nominală: ~{round(nominal_scale, 2)} m")
    print(f"  • CRS nativ: {crs}")
    print("-" * 60)

# Verificare pentru sursele din proiect:
inspect_gee_asset('LANDSAT/LC08/C02/T1_L2', is_collection=True)
inspect_gee_asset('COPERNICUS/S2_SR_HARMONIZED', is_collection=True)
inspect_gee_asset('WorldPop/GP/100m/pop', is_collection=True)

Dataset: LANDSAT/LC08/C02/T1_L2
  • Număr total benzi: 19
  • Primele benzi: ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6']
  • Rezoluție nominală: ~30 m
  • CRS nativ: EPSG:32628
------------------------------------------------------------
Dataset: COPERNICUS/S2_SR_HARMONIZED
  • Număr total benzi: 26
  • Primele benzi: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6']
  • Rezoluție nominală: ~60 m
  • CRS nativ: EPSG:32633
------------------------------------------------------------
Dataset: WorldPop/GP/100m/pop
  • Număr total benzi: 1
  • Primele benzi: ['population']
  • Rezoluție nominală: ~92.77 m
  • CRS nativ: EPSG:4326
------------------------------------------------------------


In [6]:
!pip install osmnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 3.0 MB/s eta 0:00:00


In [8]:
import osmnx as ox
import geopandas as gpd
import pandas as pd

# ==========================================
# 1. Inspectare date OpenStreetMap (OSM)
# ==========================================
print("Interogare OpenStreetMap pentru eșantionul București...")
coords = (44.4355, 26.1025) # Centru București

# Descarcă amprente clădiri pe un eșantion de 500m
gdf_buildings = ox.features_from_point(coords, tags={'building': True}, dist=500)

# Păstrăm doar poligoanele valide
gdf_buildings = gdf_buildings[gdf_buildings.geometry.type.isin(['Polygon', 'MultiPolygon'])]

# Calculăm statistici descriptive specifice OSM
# Proiectăm temporar în Stereo 70 (EPSG:3844) pentru calcule metrice reale
gdf_proj = gdf_buildings.to_crs(epsg=3844)
areas = gdf_proj.geometry.area

print("=" * 60)
print("METADATE OPENSTREETMAP (OSM) - Clădiri (Vectorial):")
print(f"  • Tip geometrie: {gdf_buildings.geometry.type.unique()}")
print(f"  • Număr obiecte în eșantion: {len(gdf_buildings)}")
print(f"  • CRS descărcat nativ: {gdf_buildings.crs}")
print(f"  • Aria medie a unei clădiri: {areas.mean():.1f} m² (Min: {areas.min():.1f} m², Max: {areas.max():.1f} m²)")
print(f"  • Coloane/Atribute principale OSM disponibile:")
for col in ['building', 'building:levels', 'height', 'addr:street']:
    if col in gdf_buildings.columns:
        valid_cnt = gdf_buildings[col].dropna().count()
        print(f"     - {col}: {valid_cnt} valori nenule ({round(valid_cnt/len(gdf_buildings)*100, 1)}%)")
print("=" * 60)

Interogare OpenStreetMap pentru eșantionul București...
METADATE OPENSTREETMAP (OSM) - Clădiri (Vectorial):
  • Tip geometrie: ['Polygon']
  • Număr obiecte în eșantion: 1009
  • CRS descărcat nativ: epsg:4326
  • Aria medie a unei clădiri: 441.6 m² (Min: 0.8 m², Max: 11909.6 m²)
  • Coloane/Atribute principale OSM disponibile:
     - building: 1009 valori nenule (100.0%)
     - building:levels: 256 valori nenule (25.4%)
     - height: 28 valori nenule (2.8%)
     - addr:street: 226 valori nenule (22.4%)


In [9]:
import ee

ee.Initialize(project='earth-engine-uhi')

# Copernicus GHSL Built-Up Grid
ghsl_asset_id = 'JRC/GHSL/P2016/BUILT_LDSMT_GLOBE_V1'
ghsl_img = ee.Image(ghsl_asset_id)

# Lista tuturor benzilor disponibile
bands = ghsl_img.bandNames().getInfo()

# Selectăm explicit prima bandă pentru a interoga proiecția și rezoluția
first_band = ghsl_img.select(0)
proj = first_band.projection()
scale = proj.nominalScale().getInfo()
crs = proj.crs().getInfo()

print("=" * 60)
print("METADATE COPERNICUS GHSL (Fond Construit):")
print(f"  • Asset ID: {ghsl_asset_id}")
print(f"  • Benzi: {bands}")
print(f"  • Rezoluție nominală: ~{round(scale, 1)} metri")
print(f"  • Sistem de coordonate: {crs}")
print("=" * 60)

METADATE COPERNICUS GHSL (Fond Construit):
  • Asset ID: JRC/GHSL/P2016/BUILT_LDSMT_GLOBE_V1
  • Benzi: ['built', 'cnfd', 'dm']
  • Rezoluție nominală: ~38.2 metri
  • Sistem de coordonate: EPSG:3857


In [10]:
import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon, Point

# 1. Bounding box general București (WGS84)
bbox_wgs84 = (25.96, 44.33, 26.24, 44.55)

# 2. Conversie în Stereo 70 (EPSG:3844) pentru coordonate metrice reale
gdf_bbox = gpd.GeoDataFrame(
    geometry=[Polygon.from_bounds(*bbox_wgs84)], 
    crs="EPSG:4326"
).to_crs("EPSG:3844")

minx, miny, maxx, maxy = gdf_bbox.total_bounds
cell_size = 250  # 250 metri

x_coords = np.arange(minx, maxx, cell_size)
y_coords = np.arange(miny, maxy, cell_size)

polygons = []
cell_ids = []
idx = 0

for x in x_coords:
    for y in y_coords:
        poly = Polygon([
            (x, y),
            (x + cell_size, y),
            (x + cell_size, y + cell_size),
            (x, y + cell_size)
        ])
        polygons.append(poly)
        cell_ids.append(f"cell_{idx:05d}")
        idx += 1

gdf_grid = gpd.GeoDataFrame({
    'cell_id': cell_ids,
    'geometry': polygons
}, crs="EPSG:3844")

# 3. Delimitarea celor două zone de comparație (Exemplu: Centru vs Periferie sau Nord vs Sud)
# Definim centrul istoric / Piața Universității în Stereo 70
punct_centru_wgs84 = gpd.GeoDataFrame(
    geometry=[Point(26.1025, 44.4355)], crs="EPSG:4326"
).to_crs("EPSG:3844").geometry.iloc[0]

# Calculăm distanța fiecărui centroid de celulă până la centru
distante_centru = gdf_grid.geometry.centroid.distance(punct_centru_wgs84)

# ZONA A: Nucleu Urban Central (buffer < 4.5 km - densitate maximă, clădiri compacte)
# ZONA B: Coridor Periurban / Exterior (distanță > 4.5 km - mixt rezidențial, industrial, spații deschise)
gdf_grid['zone_label'] = np.where(distante_centru <= 4500, 'Nucleu_Urban_Central', 'Periferie_Periurban')

# Alternativă simplă: Împărțire Nord vs Sud folosind mediana axei Y
# y_mid = (miny + maxy) / 2
# gdf_grid['zone_label'] = np.where(gdf_grid.geometry.centroid.y >= y_mid, 'Bucuresti_Nord', 'Bucuresti_Sud')

# Versiunea în WGS84 necesară pentru interogarea Google Earth Engine
gdf_grid_wgs84 = gdf_grid.to_crs("EPSG:4326")

print("Distribuția celulelor pe cele două zone de comparație:")
print(gdf_grid['zone_label'].value_counts())

Distribuția celulelor pe cele două zone de comparație:
zone_label
Periferie_Periurban     7994
Nucleu_Urban_Central    1015
Name: count, dtype: int64


In [11]:
import ee
import pandas as pd
import json

ee.Initialize(project='earth-engine-uhi')

# 1. Definim compozitul sezonier de vară 2023 (Iun - Aug)
start_date = '2023-06-01'
end_date = '2023-08-31'
aoi = ee.Geometry.BBox(*bbox_wgs84)

# --- A. Landsat 8/9 -> LST (°C) ---
def prep_landsat_lst(img):
    lst_c = img.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('lst_celsius')
    qa = img.select('QA_PIXEL')
    mask = qa.bitwiseAnd(1 << 4).eq(0).And(qa.bitwiseAnd(1 << 3).eq(0)) # fără nori/umbre
    return lst_c.updateMask(mask)

landsat_comp = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                .filterBounds(aoi)
                .filterDate(start_date, end_date)
                .filter(ee.Filter.lt('CLOUD_COVER', 30))
                .map(prep_landsat_lst)
                .median())

# --- B. Sentinel-2 -> NDVI & NDBI ---
def prep_sentinel_indices(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('ndvi')
    ndbi = img.normalizedDifference(['B11', 'B8']).rename('ndbi')
    scl = img.select('SCL')
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)) # fără nori
    return ndvi.addBands(ndbi).updateMask(mask)

sentinel_comp = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterBounds(aoi)
                 .filterDate(start_date, end_date)
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 25))
                 .map(prep_sentinel_indices)
                 .median())

# --- C. WorldPop 100m -> Populație ---
worldpop = (ee.ImageCollection('WorldPop/GP/100m/pop')
            .filterBounds(aoi)
            .filter(ee.Filter.eq('year', 2020))
            .first()
            .rename('population_count'))

# Imaginea unificată cu toate cele 4 variabile raster
composite_all = landsat_comp.addBands(sentinel_comp).addBands(worldpop)

# 2. Extracție pe loturi (batch-uri de 500 celule pentru stabilitate)
batch_size = 500
records = []
total_cells = len(gdf_grid_wgs84)

print(f"Începe extragerea din Earth Engine pentru {total_cells} de celule...")

for i in range(0, total_cells, batch_size):
    batch_gdf = gdf_grid_wgs84.iloc[i : i + batch_size]
    
    # Conversie GeoDataFrame -> ee.FeatureCollection
    geojson_features = json.loads(batch_gdf[['cell_id', 'geometry']].to_json())['features']
    ee_fc = ee.FeatureCollection(geojson_features)
    
    # Reducere zonală prin medie
    reduced = composite_all.reduceRegions(
        collection=ee_fc,
        reducer=ee.Reducer.mean(),
        scale=30
    )
    
    # Colectare rezultate
    feats = reduced.getInfo()['features']
    for f in feats:
        props = f['properties']
        records.append({
            'cell_id': props.get('cell_id'),
            'lst_celsius': props.get('lst_celsius'),
            'ndvi': props.get('ndvi'),
            'ndbi': props.get('ndbi'),
            'population_count': props.get('population_count')
        })
    print(f"Progres: {min(i + batch_size, total_cells)} / {total_cells} celule procesate.")

df_raster = pd.DataFrame(records)
print("Extragere completă din Earth Engine!")
display(df_raster.head())

Începe extragerea din Earth Engine pentru 9009 de celule...
Progres: 500 / 9009 celule procesate.
Progres: 1000 / 9009 celule procesate.
Progres: 1500 / 9009 celule procesate.
Progres: 2000 / 9009 celule procesate.
Progres: 2500 / 9009 celule procesate.
Progres: 3000 / 9009 celule procesate.
Progres: 3500 / 9009 celule procesate.
Progres: 4000 / 9009 celule procesate.
Progres: 4500 / 9009 celule procesate.
Progres: 5000 / 9009 celule procesate.
Progres: 5500 / 9009 celule procesate.
Progres: 6000 / 9009 celule procesate.
Progres: 6500 / 9009 celule procesate.
Progres: 7000 / 9009 celule procesate.
Progres: 7500 / 9009 celule procesate.
Progres: 8000 / 9009 celule procesate.
Progres: 8500 / 9009 celule procesate.
Progres: 9000 / 9009 celule procesate.
Progres: 9009 / 9009 celule procesate.
Extragere completă din Earth Engine!


,cell_id,lst_celsius,ndvi,ndbi,population_count
0,cell_00000,32.790810,0.704435,-0.204227,1.588619
1,cell_00001,34.102888,0.631350,-0.117332,1.914225
2,cell_00002,36.297801,0.554015,-0.053420,1.353452
3,cell_00003,37.699596,0.414859,-0.007600,1.095663
4,cell_00004,37.388016,0.405465,0.021421,1.712950


In [12]:
# Unim datele extrase din GEE cu grid-ul de bază
gdf_merged = gdf_grid.merge(df_raster, on='cell_id', how='left')

In [13]:
gdf_merged

,cell_id,geometry,zone_label,lst_celsius,ndvi,ndbi,population_count
0,cell_00000,"POLYGON ((576395.486 314923.984, 576645.486 31...",Periferie_Periurban,32.790810,0.704435,-0.204227,1.588619
1,cell_00001,"POLYGON ((576395.486 315173.984, 576645.486 31...",Periferie_Periurban,34.102888,0.631350,-0.117332,1.914225
2,cell_00002,"POLYGON ((576395.486 315423.984, 576645.486 31...",Periferie_Periurban,36.297801,0.554015,-0.053420,1.353452
3,cell_00003,"POLYGON ((576395.486 315673.984, 576645.486 31...",Periferie_Periurban,37.699596,0.414859,-0.007600,1.095663
4,cell_00004,"POLYGON ((576395.486 315923.984, 576645.486 31...",Periferie_Periurban,37.388016,0.405465,0.021421,1.712950
...,...,...,...,...,...,...,...
9004,cell_09004,"POLYGON ((598895.486 338423.984, 599145.486 33...",Periferie_Periurban,38.675767,0.426050,0.040955,0.415174
9005,cell_09005,"POLYGON ((598895.486 338673.984, 599145.486 33...",Periferie_Periurban,36.927706,0.545064,-0.066669,0.264648
9006,cell_09006,"POLYGON ((598895.486 338923.984, 599145.486 33...",Periferie_Periurban,37.442498,0.427111,0.053339,0.240871
9007,cell_09007,"POLYGON ((598895.486 339173.984, 599145.486 33...",Periferie_Periurban,39.798195,0.253948,0.174268,0.228021


In [14]:
import ee
import json
import pandas as pd
import geopandas as gpd

# 1. Calculăm centroizii corect în CRS metric (Stereo 70) pentru acuratețe, apoi reproiectăm în WGS84
gdf_centroids = gdf_grid[['cell_id', 'geometry']].copy()
gdf_centroids['geometry'] = gdf_centroids.geometry.centroid
gdf_centroids_wgs84 = gdf_centroids.to_crs(epsg=4326)

# 2. Preluăm stratul GHSL
ghsl_img = ee.Image('JRC/GHSL/P2016/BUILT_LDSMT_GLOBE_V1').select('built')

# 3. Eșantionare pe loturi folosind .sampleRegions()
batch_size = 1500
records_morpho = []
total_cells = len(gdf_centroids_wgs84)

print(f"Eșantionare stabilă GHSL pentru {total_cells} puncte...")

for i in range(0, total_cells, batch_size):
    batch_gdf = gdf_centroids_wgs84.iloc[i : i + batch_size]
    geojson_features = json.loads(batch_gdf.to_json())['features']
    ee_fc = ee.FeatureCollection(geojson_features)
    
    # sampleRegions este conceput nativ pentru puncte/centroizi
    sampled = ghsl_img.sampleRegions(
        collection=ee_fc,
        properties=['cell_id'],
        scale=38,
        geometries=False
    )
    
    feats = sampled.getInfo()['features']
    for f in feats:
        props = f['properties']
        records_morpho.append({
            'cell_id': props.get('cell_id'),
            'built_surface': props.get('built', 0)
        })
    print(f"Progres: {min(i + batch_size, total_cells)} / {total_cells}")

df_morpho = pd.DataFrame(records_morpho)

print("\nStatistici verificate Copernicus GHSL:")
print(df_morpho['built_surface'].describe())

Eșantionare stabilă GHSL pentru 9009 puncte...
Progres: 1500 / 9009
Progres: 3000 / 9009
Progres: 4500 / 9009
Progres: 6000 / 9009
Progres: 7500 / 9009
Progres: 9000 / 9009
Progres: 9009 / 9009

Statistici verificate Copernicus GHSL:
count    9009.000000
mean        3.301476
std         1.790144
min         1.000000
25%         2.000000
50%         2.000000
75%         6.000000
max         6.000000
Name: built_surface, dtype: float64


In [15]:

import pandas as pd
import geopandas as gpd

# 1. Îmbinăm datele morfologice GHSL cu tabelul principal extras anterior
df_final = gdf_merged.merge(df_morpho, on='cell_id', how='left')

# 2. Imputare logică pentru eventuale valori lipsă
df_final['built_surface'] = df_final['built_surface'].fillna(1).astype(int)
df_final['population_count'] = df_final['population_count'].fillna(0.0)
df_final['time_window'] = '2023_summer'

# 3. Ordonarea standard a coloanelor
cols_final = [
    'cell_id', 'time_window', 'zone_label',
    'lst_celsius', 'ndvi', 'ndbi',
    'built_surface', 'population_count',
    'geometry'
]
df_final = df_final[cols_final]

# 4. Eliminarea celulelor fără observații termice valide (mască nori / apă completă)
df_clean = df_final.dropna(subset=['lst_celsius']).copy()

# 5. Export fișiere
csv_filename = "dataset_aligned_bucharest_250m.csv"
gpkg_filename = "dataset_aligned_bucharest_250m.gpkg"

df_clean.drop(columns=['geometry']).to_csv(csv_filename, index=False)
df_clean.to_file(gpkg_filename, driver="GPKG")

print(f"Salvat cu succes: {csv_filename} ({len(df_clean)} rânduri)")
print(f"Salvat cu succes: {gpkg_filename}")

Salvat cu succes: dataset_aligned_bucharest_250m.csv (8584 rânduri)
Salvat cu succes: dataset_aligned_bucharest_250m.gpkg


In [16]:
total_teoretic = len(df_final)
total_valid = len(df_clean)
pierderi = total_teoretic - total_valid
rata_supravietuire = (total_valid / total_teoretic) * 100

print("=" * 65)
print("RAPORT DE ACOPERIRE ȘI SUPRAVIEȚUIRE (ATTRITION REPORT)")
print("=" * 65)
print(f"Unități inițiale generate în grid (250m x 250m):  {total_teoretic}")
print(f"Unități validate după fuziune și filtrare LST:   {total_valid} ({rata_supravietuire:.2f}%)")
print(f"Unități eliminate (mască de nori/apă/margini):     {pierderi} ({100 - rata_supravietuire:.2f}%)")
print("-" * 65)
print("Distribuția unităților pe zone de analiză:")
for zona, cnt in df_clean['zone_label'].value_counts().items():
    print(f"  • {zona}: {cnt} celule ({cnt/total_valid*100:.1f}%)")
print("-" * 65)
print("Sumar statistic variabile (Mediană / Medie / Deviere Standard):")
display(df_clean[['lst_celsius', 'ndvi', 'ndbi', 'built_surface', 'population_count']].describe().round(2))
print("=" * 65)

RAPORT DE ACOPERIRE ȘI SUPRAVIEȚUIRE (ATTRITION REPORT)
Unități inițiale generate în grid (250m x 250m):  9009
Unități validate după fuziune și filtrare LST:   8584 (95.28%)
Unități eliminate (mască de nori/apă/margini):     425 (4.72%)
-----------------------------------------------------------------
Distribuția unităților pe zone de analiză:
  • Periferie_Periurban: 7570 celule (88.2%)
  • Nucleu_Urban_Central: 1014 celule (11.8%)
-----------------------------------------------------------------
Sumar statistic variabile (Mediană / Medie / Deviere Standard):


,lst_celsius,ndvi,ndbi,built_surface,population_count
count,8584.00,8584.00,8584.00,8584.00,8584.00
mean,39.01,0.38,-0.01,3.31,23.30
std,3.18,0.16,0.12,1.80,31.38
min,27.04,-0.22,-0.50,1.00,0.00
25%,37.55,0.27,-0.07,2.00,2.04
50%,39.40,0.35,-0.00,2.00,7.07
75%,41.07,0.45,0.07,6.00,32.93
max,50.06,0.86,0.31,6.00,107.88


In [ ]:
# Harta interactivă a Bounding Box-ului peste fundalul real al Bucureștiului
m = gdf_bbox.explore(
    color="crimson",
    style_kwds={"fillColor": "crimson", "fillOpacity": 0.15, "weight": 2.5},
    name="Bounding Box București"
)

# Afișează harta direct în notebook
m

In [17]:
# Vizualizare interactivă a gridului colorat pe zone
gdf_grid_wgs84.sample(1500).explore(
    column="zone_label",
    cmap="coolwarm",
    style_kwds={"fillOpacity": 0.4, "weight": 0.5},
    legend=True
)